# W1D3：函数封装——把逻辑打包复用

**今天要解决的问题**：你昨天写了很多重复的订单分析代码——统计总金额、筛选大单、按城市分类……每段代码都差不多但又有点不一样。今天学怎么把重复逻辑"打包"成函数，写一次，到处用。

**你要用到的武器**：`def`函数定义 + 参数系统 + 作用域 + lambda & 推导式

**需要的前置知识**：
- W1D1：容器（list/dict/set/tuple）—— 函数操作的对象
- W1D2：控制流（if/for/while）—— 函数体内的"肌肉"

---

### 📋 今日知识地图

学完今天的内容后，你应该能逐项打勾：

**函数定义与调用**
- [ ] 用 `def` 定义函数，括号+冒号+缩进体
- [ ] 理解"定义时不执行，调用时才执行"（记菜谱 vs 做菜）
- [ ] 用 `return` 返回值（没有return → 返回None）
- [ ] 多返回值：`return a, b`（实际返回tuple）
- [ ] 写文档字符串 `"""做什么"""`（紧接着def下面）

**参数全系**
- [ ] 位置参数：`def f(a, b):` → `f(1, 2)` 按顺序对应
- [ ] 默认参数：`def f(a, b=10):` → 默认值必须在后面
- [ ] 关键字参数：`f(a=1, b=2)` 可以换顺序
- [ ] `*args` 可变位置参数（打包成tuple）
- [ ] `**kwargs` 可变关键字参数（打包成dict）

**作用域 LEGB**
- [ ] Local → Enclosing → Global → Built-in（查找顺序）
- [ ] 函数内变量外面访问不到
- [ ] 函数内可读全局变量但不可直接修改（需 `global`）
- [ ] 在函数里 `x = 10` 创建的是局部变量，不是修改全局x

**lambda & 函数式工具**
- [ ] lambda 一句话函数：`lambda x: x**2`
- [ ] `map(func, iterable)` → 每个元素套func
- [ ] `filter(func, iterable)` → 保留func返回True的
- [ ] `sorted(iterable, key=func)` → 按func返回值排序
- [ ] 返回的是迭代器（惰性求值），需 `list()` 转换

**推导式**
- [ ] 列表推导：`[x**2 for x in range(10) if x%2==0]`
- [ ] 字典推导：`{k: v*2 for k,v in d.items()}`
- [ ] 集合推导：`{x for x in lst if x>0}`
- [ ] 推导式 vs map/filter：推导式更Pythonic、可读性更好

---


## 🔴 先别学——你先撞个墙

小王让你分析三份订单数据：上周的、这周的、促销活动的。你昨天写的代码是这样的：

```python
# 分析上周订单
total_1 = 0
big_count_1 = 0
for order in last_week_orders:
    total_1 += order['金额']
    if order['金额'] > 500:
        big_count_1 += 1
print(f"上周总金额: {total_1}, 大单数: {big_count_1}")

# 分析本周订单
total_2 = 0
big_count_2 = 0
for order in this_week_orders:
    total_2 += order['金额']
    if order['金额'] > 500:
        big_count_2 += 1
print(f"本周总金额: {total_2}, 大单数: {big_count_2}")

# 分析促销订单  ← 还在复制粘贴……
total_3 = 0
big_count_3 = 0
...
```

你写了三份**几乎一模一样**的代码，只改了数据来源和变量名。

**问题**：如果小王明天又给你 10 个城市的独立数据，让你各自统计——你打算复制粘贴 10 遍吗？如果不复制，你还有别的办法吗？

试一下：用你目前学的知识（变量、容器、for、if），你能把上面那段逻辑"打包"成一个可以反复使用的东西吗？

> 你刚才撞到的这个墙——"重复逻辑怎么打包"，今天要学的**函数**就是专门解决它的。


## 📦 回到速卖通

昨天你用 for + if 帮小王分析了订单数据，统计了大单数、总金额、平均值。小王看了你的代码后说：

> "同学，代码写得对，但我看一眼就发现有三段代码长得几乎一模一样——统计总金额那段、统计大单数那段、找最大单那段。你有没有办法把'统计'这个动作抽象出来，别每次都重新写一遍？"

今天你的任务：学会把重复逻辑**封装成函数**，写一次，任何数据来了都能直接用。


---

## 概念一：函数是什么

### 🤔 你现有的知识够用吗？

你现在只会一种"执行代码"的方式：从上到下，一行一行写。

但现实中呢？你妈教你怎么煮面——她只教一次"食谱"，你以后每次煮面都按这个食谱来，不需要每次都重新教一遍。

那在Python里呢？你现在写了三遍统计逻辑，每次只是数据不同，逻辑完全一样。

**如果你能把"统计逻辑"写成一份"食谱"，每次只要说"按食谱A处理这份数据"，代码量减少2/3，而且改逻辑只用改一处——这就是函数。**

你刚才撞到的这个墙，**函数**就是专门解决它的。


### 函数定义与调用

**① 说白了就是**：函数就像一份菜谱。`def` 是"把菜谱写下来"——这时你还没做菜。`函数名(数据)` 是"按菜谱做菜"——这时你才动手。菜谱写一次，可以反复做无数次。

**② 在沙盘里怎么用**：你把"遍历订单list → 累加金额 → 返回总和"写成一个叫 `calc_total` 的函数。以后不管是上周的订单、本周的订单、还是促销订单，你只要 `calc_total(任意订单list)` 就行，不用每次重新写循环。

**③ 跟你昨天学的 for/if 的关系**：函数是壳，for/if 是肉。函数体里面主要就是 for 和 if —— 昨天你学会了"循环和判断"，今天你把它们装进函数这个"盒子"里，盒子外面贴个标签（函数名），以后按标签取用。

**④ 为什么这样设计**：
- **为什么定义时不执行？** 如果每次读到 `def` 就执行，那每个 Python 脚本启动时会把所有函数跑一遍——浪费且危险。"定义和执行分离"让你可以控制何时执行。
- **为什么 `return` 不能省？** 没有 return 的函数自动返回 `None`。很多 bug 来自"我明明调了函数但结果是 None"——因为你忘了写 return 语句。
- **多返回值为什么返回 tuple？** `return a, b` 实际上是 `return (a, b)` —— Python 自动打包成 tuple。你可以用 `x, y = func()` 解包，这个语法在 D1 学过多重赋值。
- **文档字符串的约定**：`"""..."""` 写在函数体内第一行，Python 把它存在 `func.__doc__` 里。`help(func)` 能自动显示它——这比注释强，因为注释会被解释器忽略，但 docstring 是运行时可见的。


In [1]:
# 定义函数：写菜谱（这时什么都不发生）
def calc_total(orders):
    """计算订单list的总金额"""
    total = 0
    for order in orders:
        total += order['金额']
    return total          # 别忘了return！

# 速卖通订单数据
orders_a = [
    {"订单号": "ORD001", "商品": "蓝牙耳机", "金额": 299},
    {"订单号": "ORD002", "商品": "运动鞋", "金额": 159},
    {"订单号": "ORD003", "商品": "充电宝", "金额": 89},
]

# 调用函数：做菜
result = calc_total(orders_a)
print(f"总金额: ¥{result}")  # 总金额: ¥547

# 同一份菜谱，换一份数据也能用
orders_b = [{"订单号": "ORD004", "商品": "键盘", "金额": 320}]
print(calc_total(orders_b))  # 320 —— 不用重新写循环！


总金额: ¥547
320


In [3]:
# 【函数小练】
# 定义一个函数 count_big_orders(orders, threshold)，
# 让它返回"金额大于threshold的订单数量"
# 然后用下面的数据测试：

orders = [
    {"订单号": "ORD001", "金额": 299},
    {"订单号": "ORD002", "金额": 680},
    {"订单号": "ORD003", "金额": 520},
    {"订单号": "ORD004", "金额": 89},
]

# 你的代码写在这里：

def count_big_orders(orders,threshold):
    count = 0 
    for order in orders :
        if order["金额"] > threshold:
            count += 1
    return count
result = count_big_orders(orders, 500)
print(f"count_big_orders(orders, 500) → {result}")
result = count_big_orders(orders, 300)
print(f"count_big_orders(orders, 300) → {result}")
        
# 预期输出：count_big_orders(orders, 500) → 2
# 预期输出：count_big_orders(orders, 300) → 3


count_big_orders(orders, 500) → 2
count_big_orders(orders, 300) → 2


---

## 概念二：参数——函数的"入口"

### 🤔 你的 `calc_total` 够灵活吗？

你现在写的 `calc_total(orders)` 只能算总金额。如果小王说"这个函数能不能也支持统计大单数？"——你可能会写一个新函数 `calc_big_count(orders)`。

但如果小王又说"能不能支持按指定品类统计？" "能不能按指定城市统计？"——你难道写无数个函数？

**问题**：怎么让一个函数灵活应对不同需求，而不是每个需求写一个新函数？

你刚才撞到的这个墙——"函数怎么变灵活"，**参数系统**就是专门解决它的。


### 参数全系

**① 说白了就是**：参数是函数的"遥控器按钮"。默认参数是"有个预设值，你不调就用预设"。关键字参数是"按钮上有标签，你可以跳过不用的按钮直接调要改的那个"。`*args` 是"我不确定你要传几个东西，多多益善"。`**kwargs` 是"我不确定你要传什么标签的东西"。

**② 在沙盘里怎么用**：
- 位置参数：`def analyze(orders, city)` → 必须传两个值
- 默认参数：`def analyze(orders, threshold=500)` → 不传 threshold 就默认 500 元为大单
- 关键字参数：`analyze(orders, threshold=300)` 或 `analyze(orders, city="北京")` 可跳过中间参数
- `*args`：`def merge_orders(*order_lists)` → 你传几个 list 都行，它们被打包成一个 tuple

**③ 跟你第一天学的容器（tuple/dict）的关系**：`*args` 把多出来的位置参数打包成 **tuple**，`**kwargs` 把多出来的关键字参数打包成 **dict**。你看，D1 学的东西出现了——tuple 不可修改正好适合"收到的参数不让你改"这个语义。

**④ 为什么这样设计**：
- **默认值必须在后面**：因为 Python 按位置匹配参数。如果 `def f(a=1, b):` 你写 `f(2)`，Python 不知道 2 是给 a 还是给 b。所以规则是：**有默认值的参数必须排在没默认值的后面**。
- **默认参数只在定义时求值一次** —— 这是 Python 最出名的陷阱之一（详见陷阱专题）。`def f(lst=[]):` 的空列表在函数定义时就创建了，之后每次调用都共享同一个列表对象。
- **关键字参数让你跳过中间参数**：`def f(a, b=10, c=20):` 你可以 `f(1, c=30)` —— b 用默认值 10，c 覆盖为 30。你不会想到在只有位置参数的语言里跳过中间参数有多麻烦。


In [4]:
# === 位置参数 + 默认参数 + 关键字参数 ===
def judge_order(amount, threshold=500, strict=False):
    """判断订单大小
    amount:   订单金额
    threshold: 大单门槛（默认500）
    strict:   严格模式——等于门槛算大单（默认False，等于算小单）
    """
    if strict:
        return "大单" if amount >= threshold else "小单"
    else:
        return "大单" if amount > threshold else "小单"

# 三种调用方式
print(judge_order(680))                        # 大单（用默认threshold=500）
print(judge_order(680, threshold=300))         # 大单（threshold改成300）
print(judge_order(500, strict=True))           # 大单（严格模式，等于也算大单）
print(judge_order(500, threshold=300, strict=True))  # 大单（关键字参数可换顺序）

# === *args：不固定数量 ===
def merge_sales(*city_sales):
    """合并多个城市的销售额list，返回总和"""
    total = 0
    for sales in city_sales:   # city_sales 是 tuple: ([...], [...], ...)
        total += sum(sales)
    return total

print(merge_sales([100, 200], [300, 400]))          # 1000
print(merge_sales([50], [60], [70], [80]))          # 260


大单
大单
大单
大单
1000
260


In [4]:
# 【参数小练】
# 定义一个函数 filter_orders(orders, city=None, min_amount=0)
# - 如果传了city，只返回该城市的订单
# - 如果传了min_amount，只返回金额>=min_amount的订单
# - city和min_amount都可以不传（默认不过滤）
# 两个过滤条件可以同时生效

orders = [
    {"订单号": "ORD001", "城市": "北京", "金额": 299},
    {"订单号": "ORD002", "城市": "上海", "金额": 680},
    {"订单号": "ORD003", "城市": "北京", "金额": 520},
    {"订单号": "ORD004", "城市": "深圳", "金额": 89},
    {"订单号": "ORD005", "城市": "北京", "金额": 45},
]

# 你的代码写在这里：
def filter_orders(orders,city=None,min_amount=0):
    filtered_orders = []
    for order in orders:
        if city is None or order["城市"] == city:
            if min_amount <= order["金额"]:
                filtered_orders.append(order)
    return filtered_orders

result = filter_orders(orders, city="北京")
print(f"filter_orders(orders, city=\"北京\") → {len(result)}条订单")
result = filter_orders(orders, min_amount=500)
print(f"filter_orders(orders, min_amount=500) → {len(result)}条订单 {[order['订单号'] for order in result]}")
result = filter_orders(orders, city="北京", min_amount=500)
print(f"filter_orders(orders, city=\"北京\", min_amount=500) → {len(result)}条订单 {[order['订单号'] for order in result]}")


# 预期：filter_orders(orders, city="北京") → 3条订单
# 预期：filter_orders(orders, min_amount=500) → 2条订单(ORD002, ORD003)
# 预期：filter_orders(orders, city="北京", min_amount=500) → 1条订单(ORD003)


filter_orders(orders, city="北京") → 3条订单
filter_orders(orders, min_amount=500) → 2条订单 ['ORD002', 'ORD003']
filter_orders(orders, city="北京", min_amount=500) → 1条订单 ['ORD003']


---

## 概念三：作用域——函数内外的"墙"

### 🤔 为什么外面读不到函数里的变量？

运行下面这段代码（在你心里，先别跑）：

```python
def calc():
    result = 100
    return result

calc()
print(result)  # ← 会打印 100 吗？
```

答案是：**报错！`NameError: name 'result' is not defined`**

你明明在函数里创建了 `result`，为什么外面用不了？反过来——函数里能改外面定义的变量吗？

你刚才撞到的这个墙——"变量到底在哪可见"，**作用域规则**就是专门解释它的。


### 作用域 LEGB

**① 说白了就是**：函数有"墙壁"——函数里面创建的变量外面看不见，就像你在自己房间里放的东西，隔壁邻居拿不到。反过来，函数可以看到客厅里的东西（全局变量），但不能擅自移动客厅的家具（修改全局变量需要特殊权限——`global`声明）。

**② 在沙盘里怎么用**：你的 `calc_total` 函数里创建的 `total` 变量只活在函数执行期间——函数一结束，`total` 就被回收了。如果多个函数需要共享数据（比如订单税率），把它定义在函数外面（全局），函数里只读不写。

**③ 跟你昨天学的 if/for 的关系**：if 和 for 块**不创建作用域**！在 if 块里定义的变量，出了 if 还能用。但函数里定义的变量，出了函数就没了。这是 Python 的一个特殊规则——只有 `def` / `class` / `lambda` 创建新的作用域。

**④ 为什么这样设计——LEGB 查找顺序**：
- **L (Local)**：函数内部定义的变量。最先查。
- **E (Enclosing)**：外层函数定义的变量（嵌套函数时用）。
- **G (Global)**：文件顶层定义的变量。
- **B (Built-in)**：Python 内置的名字（`print`, `len`, `sum`……）。最后查。

**为什么 Python 不允许函数里直接修改全局变量？** 如果每个函数都能随便改全局变量，你写代码时永远不知道某个变量被谁改了——这是所有大规模软件的噩梦。`global` 声明是一个"我知道我在做什么"的声明，逼你想清楚再改。

**最容易犯的错**：在函数里写 `x = 10`，你以为你在修改全局的 `x`——不，你创建了一个同名的局部变量，和外边的 `x` 毫无关系。


In [5]:
# === 局部变量：函数在里面，外面看不到 ===
def make_report():
    title = "速卖通周报"   # 局部变量
    return title

print(make_report())       # 速卖通周报
# print(title)             # ❌ NameError! title只活在函数里面

# === 全局变量：能读，不能直接改 ===
TAX_RATE = 0.13            # 全局变量（常量约定全大写）

def calc_tax(amount):
    return amount * TAX_RATE  # ✅ 读全局变量，OK

print(calc_tax(100))       # 13.0

# === 想改全局变量？必须声明global ===
error_count = 0

def log_error():
    global error_count     # 告诉Python：我要用外面的error_count
    error_count += 1       # 现在才是修改外面的error_count

log_error()
log_error()
print(error_count)         # 2


速卖通周报
13.0
2


In [14]:
# 【作用域小练】
# 下面代码有bug，找出并修复：

total_sales = 2

def add_sales(amount):
    total_sales = total_sales + amount  # ← 这行报错！为什么？
    return total_sales

# 你的修复方案写在这里（两种都可以）：
# 方案1：用global声明
# 方案2：不用全局变量，用return累加
def add_sales(amount):
    global total_sales
    total_sales = total_sales + amount
    return total_sales
print(add_sales(100))
def add_sales(amount):
    return total_sales + amount
print(add_sales(200))


# 提示：报错原因是 Python 看到函数里有 total_sales = ...
# 就认为 total_sales 是局部变量，但右边又在读它（还没赋值）→ UnboundLocalError


102
302


---

## 概念四：函数的"快捷方式"—— lambda & 推导式

### 🤔 三行代码定义一个只用一次的函数——太啰嗦了吧？

看这个场景：你有一个玩家列表，想按等级排序。

```python
# 用 def —— 3行
def get_level(player):
    return player['等级']
sorted(players, key=get_level)
```

这函数就一行 `return` 逻辑，而且别的地方根本用不到 `get_level`——给它起名字、写三行，就像花 30 分钟给一个只吃一顿的泡面精雕细刻摆盘。

**Python 有没有"用完就扔"的函数写法？能不能一行搞定？**

你刚才撞到的这个墙——"只用一次的函数怎么写得快"，**lambda** 和**推导式**就是专门解决它的。


### lambda & 推导式

**① 说白了就是**：
- **lambda** 是"一次性筷子"——用完就扔的函数，不用起名字。`lambda x: x['等级']` 翻译成人话："喂一个东西进来，返回它的'等级'字段"。
- **map** 是流水线工人——你给他一个操作，他对每个零件做一遍，传给你下一个零件。
- **filter** 是质检员——你给他一个检验标准，他把不合格的扔掉，只留合格的。
- **sorted(key=...)** 是按你说的标准排序——你给它一个"怎么看大小"的函数。
- **推导式** 是把 map+filter 写得更像自然语言—— `[给每个东西做什么 for 东西 in 一堆东西 if 条件]`。

**② 在沙盘里怎么用**：
- `sorted(orders, key=lambda o: o['金额'], reverse=True)` → 订单按金额降序排列
- `list(filter(lambda o: o['金额'] > 500, orders))` → 筛出所有大单
- `list(map(lambda o: o['金额'], orders))` → 提取所有订单金额
- `[o['订单号'] for o in orders if o['金额'] > 500]` → 一行代码拿大单号

**③ 跟你两天前学的 for 循环的关系**：`map`、`filter`、推导式做的事，for 循环全能做——但区别在哪？
- for 循环：你一步步走，Python 一步步执行——**命令式**（你告诉 Python **怎么**做）
- map/filter/推导式：你描述"我要什么"，Python 自己去执行——**声明式**（你告诉 Python **要什么**）
- 推导式是 Python 社区的首选风格——比 map+lambda 可读性更好

**④ 为什么这样设计**：
- **map/filter 返回迭代器而非 list**：惰性求值——你不跟它要结果，它就不算。好处是处理 100 万条数据时不会一次性占满内存。需要用的时候 `list()` 一次性求值。
- **lambda 只能写表达式，不能写语句**：这是故意设计成"只用一次"的限制。`lambda x: x+1` 可以，`lambda x: print(x)` 也可以（但不要这样用），但 `lambda x: if x>0: return x` 不行——if/for/return 是语句，lambda 里只能用表达式。需要多行逻辑？老老实实用 def。
- **推导式有独立作用域（Python 3+）**：推导式内部的循环变量不会泄漏到外面。`[x for x in range(10)]` 执行完后，外面的 `x` 不变。这是 Python 3 对 Python 2 的重要修正。
- **sorted 的 key 参数为什么设计成接受函数**：你可以自定义"怎么看大小"而不用修改数据本身。key 函数对每个元素调用一次，sorted 用返回值来比较——这叫 Schwartzian transform（装饰-排序-去装饰），效率比多次调用比较函数高。


In [ ]:
# === lambda：一次性函数 ===
orders = [
    {"订单号": "ORD001", "金额": 299, "城市": "北京"},
    {"订单号": "ORD002", "金额": 680, "城市": "上海"},
    {"订单号": "ORD003", "金额": 520, "城市": "北京"},
    {"订单号": "ORD004", "金额": 89,  "城市": "深圳"},
]

# lambda 替代 def：sorted 按金额降序
top_orders = sorted(orders, key=lambda o: o['金额'], reverse=True)
print([o['订单号'] for o in top_orders])  # ['ORD002', 'ORD003', 'ORD001', 'ORD004']

# === map：每个人套同一个操作 ===
amounts = list(map(lambda o: o['金额'], orders))
print(amounts)  # [299, 680, 520, 89]

# === filter：只留符合条件的 ===
big_orders = list(filter(lambda o: o['金额'] > 500, orders))
print([o['订单号'] for o in big_orders])  # ['ORD002', 'ORD003']

# === 推导式：比 map+filter 更 Pythonic ===
big_order_ids = [o['订单号'] for o in orders if o['金额'] > 500]
print(big_order_ids)  # ['ORD002', 'ORD003'] —— 一行搞定！

# === 字典推导 ===
order_dict = {o['订单号']: o['金额'] for o in orders}
print(order_dict)  # {'ORD001': 299, 'ORD002': 680, ...}


In [10]:
# 【lambda & 推导式小练】
# 用推导式（不要用for循环）完成以下三个任务：

orders = [
    {"订单号": "ORD001", "金额": 299, "城市": "北京"},
    {"订单号": "ORD002", "金额": 680, "城市": "上海"},
    {"订单号": "ORD003", "金额": 520, "城市": "北京"},
    {"订单号": "ORD004", "金额": 89,  "城市": "深圳"},
    {"订单号": "ORD005", "金额": 45,  "城市": "广州"},
]

# 任务1（列表推导）：生成所有"金额>100"的订单号列表
# 预期：['ORD001', 'ORD002', 'ORD003']

order_ids = list(filter(lambda order: order["金额"] > 100, orders))
print([order["订单号"] for order in order_ids])


# 任务2（列表推导）：生成所有"北京"订单的金额列表
# 预期：[299, 520]

order_amounts = [order["金额"] for order in orders if order["城市"] == "北京"]
print(order_amounts)

# 任务3（字典推导）：生成 {订单号: 金额} 但只含金额>=300的订单
# 预期：{'ORD002': 680, 'ORD003': 520}

order_dict = {order["订单号"]: order["金额"] for order in orders if order["金额"] >= 300}
print(order_dict)


['ORD001', 'ORD002', 'ORD003']
[299, 520]
{'ORD002': 680, 'ORD003': 520}


In [11]:
# ========== 基础练习1：计算订单总金额 ==========
# 场景：速卖通电商 | 用到的武器：def、for、list[dict]
# 覆盖教学规格知识点：函数定义与调用、return返回值、遍历list

orders = [
    {"订单号": "ORD001", "商品": "蓝牙耳机", "品类": "数码", "金额": 299, "城市": "北京"},
    {"订单号": "ORD002", "商品": "运动鞋",   "品类": "服饰", "金额": 159, "城市": "上海"},
    {"订单号": "ORD003", "商品": "充电宝",   "品类": "数码", "金额": 89,  "城市": "深圳"},
    {"订单号": "ORD004", "商品": "连衣裙",   "品类": "服饰", "金额": 520, "城市": "北京"},
    {"订单号": "ORD005", "商品": "鼠标",     "品类": "数码", "金额": 45,  "城市": "广州"},
]

# 任务：定义函数 calc_total(orders)，返回所有订单的总金额
# 要求：
#   1. 必须有文档字符串（"""..."""）
#   2. 必须用 return 返回结果
#   3. 用 for 遍历

# 你的代码写在这里：
def calc_total(orders):
    "所有订单的金额总和"
    total = 0
    for order in orders:
        total += order["金额"]
    return total
result = calc_total(orders)
print(result)




# 调用测试：
# print(calc_total(orders))  # 预期：1112


1112


In [14]:
# ========== 基础练习2：判断订单大小 ==========
# 场景：速卖通电商 | 用到的武器：def、默认参数、if/elif/else
# 覆盖教学规格知识点：默认参数、关键字参数、return返回值

# 任务：定义函数 judge_order(amount, threshold=500)
# 返回规则：
#   amount > threshold  → 返回 "大单"
#   amount == threshold → 返回 "持平"
#   amount < threshold  → 返回 "小单"

# 要求：
#   1. threshold 默认值为 500
#   2. 分别用位置参数和关键字参数两种方式调用并打印结果

# 你的代码写在这里：
def judge_order(amount,threshold=500):
    "判断订单大小"
    if amount > threshold:
        return "大单"
    elif amount == threshold:
        return "持平"
    else:
        return "小单"
result = judge_order(680)
print(f"judge_order(680) → {result}")
result = judge_order(500)
print(f"judge_order(500) → {result}")
result = judge_order(299)
print(f"judge_order(299) → {result}")
result = judge_order(299, threshold=300)
print(f"judge_order(299, threshold=300) → {result}")





# 预期输出：
# judge_order(680)            → 大单
# judge_order(500)            → 持平
# judge_order(299)            → 小单
# judge_order(299, threshold=300) → 小单


judge_order(680) → 大单
judge_order(500) → 持平
judge_order(299) → 小单
judge_order(299, threshold=300) → 小单


In [19]:
# ========== 基础练习3：列表推导筛订单 ==========
# 场景：速卖通电商 | 用到的武器：列表推导、if条件
# 覆盖教学规格知识点：列表推导、推导式vs for循环

orders = [
    {"订单号": "ORD001", "商品": "蓝牙耳机", "品类": "数码", "金额": 299, "城市": "北京"},
    {"订单号": "ORD002", "商品": "运动鞋",   "品类": "服饰", "金额": 159, "城市": "上海"},
    {"订单号": "ORD003", "商品": "充电宝",   "品类": "数码", "金额": 89,  "城市": "深圳"},
    {"订单号": "ORD004", "商品": "连衣裙",   "品类": "服饰", "金额": 520, "城市": "北京"},
    {"订单号": "ORD005", "商品": "鼠标",     "品类": "数码", "金额": 45,  "城市": "广州"},
    {"订单号": "ORD006", "商品": "背包",     "品类": "服饰", "金额": 680, "城市": "深圳"},
    {"订单号": "ORD007", "商品": "键盘",     "品类": "数码", "金额": 320, "城市": "上海"},
]

# 任务1（列表推导）：生成所有"数码"品类的订单号列表
# 预期：['ORD001', 'ORD003', 'ORD005', 'ORD007']
order_ids = [order["订单号"] for order in orders if order["品类"] == "数码"]
print(f"所有数码品类的订单号列表：{order_ids}")


# 任务2（列表推导）：计算所有"金额>500"的订单的总金额
# 提示：把列表推导包在sum()里
order_amounts = [order["金额"] for order in orders if order["金额"] > 500]
print(f"所有金额>500的订单的总金额：{sum(order_amounts)}")

order_amounts_02 = list(filter(lambda order: order["金额"] > 500, orders))
print(f"所有金额>500的订单的总金额：{sum([order["金额"] for order in order_amounts_02])}")

# 预期：1200


# 任务3（字典推导）：生成 {订单号: 城市} 的字典，但只含"北京"或"上海"的订单
# 预期：{'ORD001': '北京', 'ORD002': '上海', 'ORD004': '北京', 'ORD007': '上海'}
order_dict = {order["订单号"]: order["城市"] for order in orders if order["城市"] in ["北京", "上海"]}
print(f"所有订单号和城市的字典：{order_dict}")


所有数码品类的订单号列表：['ORD001', 'ORD003', 'ORD005', 'ORD007']
所有金额>500的订单的总金额：1200
所有金额>500的订单的总金额：1200
所有订单号和城市的字典：{'ORD001': '北京', 'ORD002': '上海', 'ORD004': '北京', 'ORD007': '上海'}


In [28]:
# ========== 进阶练习1：玩家排行榜 ==========
# 场景：手游工作室 | 用到的武器：lambda、sorted(key=)、列表推导 [新:lambda+sorted] [旧:list[dict]]
# 覆盖教学规格知识点：lambda表达式、sorted(key=lambda)、多级排序

players = [
    {"昵称": "暗影剑客", "等级": 45, "战力": 8200, "充值额": 200},
    {"昵称": "冰霜法师", "等级": 60, "战力": 15000, "充值额": 500},
    {"昵称": "风暴弓手", "等级": 45, "战力": 9100, "充值额": 350},
    {"昵称": "疾风刺客", "等级": 72, "战力": 18500, "充值额": 100},
    {"昵称": "圣光骑士", "等级": 60, "战力": 14200, "充值额": 800},
]

# 任务1：用 sorted + lambda 将玩家按等级降序排列
# 要求：打印排序后的玩家昵称列表
# 预期：['疾风刺客', '冰霜法师', '圣光骑士', '暗影剑客', '风暴弓手']
#       （等级72 > 60=60 > 45=45；同等级的后面任务处理）
player_names = sorted(players, key=lambda pl: pl['等级'], reverse=True)
print(f"按等级降序排列的玩家昵称列表：{[player["昵称"] for player in player_names]}")


# 任务2：多级排序——先按等级降序，等级相同按充值额降序
# 提示：lambda的返回值可以是一个tuple，sorted按tuple的元素顺序比较
# 预期昵称顺序：['疾风刺客', '圣光骑士', '冰霜法师', '风暴弓手', '暗影剑客']
# 解释：疾风(72), 圣光/冰霜(60→充值800>500), 风暴/暗影(45→充值350>200)
player_names = sorted(players, key=lambda pl: (pl['等级'], pl['充值额']), reverse=True)
print(f"按等级降序排列的玩家昵称列表：{[player["昵称"] for player in player_names]}")


# 任务3：用列表推导生成"等级>=60且充值额>=300"的高价值玩家昵称列表
# 预期：['冰霜法师', '圣光骑士']
player_names = [pl["昵称"] for pl in players if pl["等级"] >= 60 and pl["充值额"] >= 300]
print(f"等级>=60且充值额>=300的高价值玩家昵称列表：{player_names}")


按等级降序排列的玩家昵称列表：['疾风刺客', '冰霜法师', '圣光骑士', '暗影剑客', '风暴弓手']
按等级降序排列的玩家昵称列表：['疾风刺客', '圣光骑士', '冰霜法师', '风暴弓手', '暗影剑客']
等级>=60且充值额>=300的高价值玩家昵称列表：['冰霜法师', '圣光骑士']


In [31]:
# ========== 进阶练习2：出行数据分析 ==========
# 场景：滴滴出行 | 用到的武器：map、filter、lambda、sum [新:map+filter] [旧:list[dict]]
# 覆盖教学规格知识点：map、filter、迭代器惰性求值

trips = [
    {"行程ID": "T001", "乘客": "张先生", "城市": "北京", "金额": 35, "里程km": 5.2},
    {"行程ID": "T002", "乘客": "李女士", "城市": "上海", "金额": 78, "里程km": 12.8},
    {"行程ID": "T003", "乘客": "王先生", "城市": "北京", "金额": 22, "里程km": 3.1},
    {"行程ID": "T004", "乘客": "赵女士", "城市": "深圳", "金额": 120, "里程km": 18.5},
    {"行程ID": "T005", "乘客": "刘先生", "城市": "北京", "金额": 55, "里程km": 8.0},
    {"行程ID": "T006", "乘客": "陈女士", "城市": "上海", "金额": 43, "里程km": 6.5},
]

# 任务1：用 map 提取所有行程的"金额" → 用 filter 筛出金额>50的 → 用 sum 算总额
# 要求：链式操作，最终一行代码完成（嵌套调用）
# 提示：map和filter返回迭代器，sum可以直接消费迭代器，不用list()
# 预期：78 + 120 + 55 = 253
total_amount = sum(map(lambda trip: trip["金额"], filter(lambda trip: trip["金额"] > 50, trips)))
print(f"所有金额>50的行程的金额总和：{total_amount:.2f}")


# 任务2：用 map + lambda 计算每单的"每公里单价"（金额/里程，保留1位小数）
# 要求：返回一个list
# 预期：[6.7, 6.1, 7.1, 6.5, 6.9, 6.6]（大致）

price_per_km = list(map(lambda trip: round(trip["金额"]/trip["里程km"], 1), trips))
print(f"所有行程的每公里单价列表：{price_per_km}")


# 任务3（思考题）：为什么map和filter外面有时要包list()有时不用？
# 提示：试试直接 print(map(...)) 看输出是什么
# 把你的理解写在注释里：
price_per_km = map(lambda trip: round(trip["金额"]/trip["里程km"], 1), trips)
print(f"所有行程的每公里单价列表：{price_per_km}")
# 解释：map返回迭代器，直接打印会显示内存地址，需要转换为list才能查看
# 而list()可以将迭代器转换为列表，方便查看和操作

所有金额>50的行程的金额总和：253.00
所有行程的每公里单价列表：[6.7, 6.1, 7.1, 6.5, 6.9, 6.6]
所有行程的每公里单价列表：<map object at 0x000002AB1B156050>


In [ ]:
# ========== 挑战练习：重构重复代码 ==========
# 场景：速卖通电商 | 用到的武器：def、参数设计、for/if、list[dict]（今天全部武器 + D2）
# 覆盖教学规格知识点：函数封装、参数设计、L4重构能力

# 下面是小王昨天写的"订单分析脚本"——大量重复逻辑
# 你的任务：至少试15分钟，把重复逻辑提取成3个函数
# 如果卡住了，在注释里写下你卡在哪一步、你的思路是什么

orders = [
    {"订单号": "ORD001", "城市": "北京", "品类": "数码", "金额": 299},
    {"订单号": "ORD002", "城市": "上海", "品类": "服饰", "金额": 159},
    {"订单号": "ORD003", "城市": "深圳", "品类": "数码", "金额": 89},
    {"订单号": "ORD004", "城市": "北京", "品类": "服饰", "金额": 520},
    {"订单号": "ORD005", "城市": "广州", "品类": "数码", "金额": 45},
    {"订单号": "ORD006", "城市": "深圳", "品类": "服饰", "金额": 680},
    {"订单号": "ORD007", "城市": "上海", "品类": "数码", "金额": 320},
    {"订单号": "ORD008", "城市": "北京", "品类": "数码", "金额": 150},
]

# ===== 小王的老代码（重复！）=====
# --- 统计全部订单 ---
total_all = 0
count_all = 0
max_all = 0
for o in orders:
    total_all += o['金额']
    count_all += 1
    if o['金额'] > max_all:
        max_all = o['金额']
print(f"全部: 总数{count_all}, 总额{total_all}, 最大单{max_all}")

# --- 统计北京订单 ---（和上面几乎一样！）
total_bj = 0
count_bj = 0
max_bj = 0
for o in orders:
    if o['城市'] == '北京':
        total_bj += o['金额']
        count_bj += 1
        if o['金额'] > max_bj:
            max_bj = o['金额']
print(f"北京: 总数{count_bj}, 总额{total_bj}, 最大单{max_bj}")

# --- 统计数码品类 ---（又是几乎一样！）
total_digi = 0
count_digi = 0
max_digi = 0
for o in orders:
    if o['品类'] == '数码':
        total_digi += o['金额']
        count_digi += 1
        if o['金额'] > max_digi:
            max_digi = o['金额']
print(f"数码: 总数{count_digi}, 总额{total_digi}, 最大单{max_digi}")

# ===== 你的重构 =====
# 要求：提取3个函数（函数名和参数自己设计）：
#   1. calc_stats(orders)          → 统计总金额、订单数、最大单金额 → 返回(总和, 数量, 最大值)
#   2. filter_by_city(orders, city) → 筛出指定城市的订单 → 返回过滤后的list
#   3. summarize(orders, label)     → 调用上面两个函数，打印一行统计结果
#
# 重构后，上面的三块重复代码应该变成：
# summarize(orders, "全部")
# summarize(filter_by_city(orders, "北京"), "北京")
# summarize(filter_by_city(orders, "数码"), "数码")    ← 这里品类用filter_by_city不对！
#                                                         你需要自己设计参数

# 你的重构代码写在这里：
def calc_stats(orders):
    total_all = 0
    max_order = 0
    count_all = 0
    
    for order in orders:
        total_all += order["金额"]
        count_all += 1
        if order["金额"] > max_order:
            max_order = order["金额"]
    
    return total_all, max_order, count_all

total_all, max_order, count_all = calc_stats(orders)
print(f"总金额：{total_all}, 最大单金额：{max_order}, 订单数：{count_all}")

def filter_by_city(orders, city=None, category=None):
   filtered_orders = []
   for order in orders:
      if city and city != order["城市"] :
        continue
      if category and category != order["品类"]:
        continue
   return filtered_orders
print(filter_by_city(orders, "北京", "数码"))


def summarize(orders, label):
    total_all, max_order, count_all = calc_stats(orders)
    print(f"{label}: 总数{count_all}, 总额{total_all}, 最大单{max_order}")



















全部: 总数8, 总额2262, 最大单680
北京: 总数3, 总额969, 最大单520
数码: 总数5, 总额903, 最大单320
总金额：2262, 最大单金额：680, 订单数：8
[{'订单号': 'ORD001', '城市': '北京', '品类': '数码', '金额': 299}, {'订单号': 'ORD004', '城市': '北京', '品类': '服饰', '金额': 520}, {'订单号': 'ORD008', '城市': '北京', '品类': '数码', '金额': 150}]
[{'订单号': 'ORD001', '城市': '北京', '品类': '数码', '金额': 299}, {'订单号': 'ORD004', '城市': '北京', '品类': '服饰', '金额': 520}, {'订单号': 'ORD008', '城市': '北京', '品类': '数码', '金额': 150}]


In [5]:
# ========== 迁移练习：外卖配送费计算 ==========
# 场景：美团外卖 | 用到的武器：你需要自己判断今天学的哪个武器能解决这个问题
# 覆盖教学规格知识点：函数定义、if/elif条件判断、参数设计

# 美团配送费规则：
#   距离 < 3km  → 配送费 ¥5
#   距离 3-5km  → 配送费 ¥8
#   距离 > 5km  → 配送费 ¥12
#
# 注意边界：3.0km算"3-5km"档，5.0km算">5km"档

orders = [
    {"订单号": "M001", "商家": "黄焖鸡米饭", "距离km": 2.1, "金额": 35},
    {"订单号": "M002", "商家": "兰州拉面",     "距离km": 4.5, "金额": 78},
    {"订单号": "M003", "商家": "沙县小吃",     "距离km": 6.0, "金额": 22},
    {"订单号": "M004", "商家": "麦当劳",       "距离km": 3.0, "金额": 120},
    {"订单号": "M005", "商家": "星巴克",       "距离km": 0.8, "金额": 55},
    {"订单号": "M006", "商家": "海底捞外卖",   "距离km": 5.0, "金额": 280},
]

# 任务1：定义函数 delivery_fee(distance)，返回配送费
# 你需要自己判断今天学的哪个武器能解决这个问题
def delivery_fee(distance):
    """
    计算配送配送费
    """
    if distance < 3:
        return 5
    elif distance <= 5:
        return 8
    else:
        return 12    

# 任务2：计算所有订单的配送费总收入
# 提示：可以用推导式 + sum
# 预期：M001(5) + M002(8) + M003(12) + M004(8) + M005(5) + M006(12) = 50
total_fee = sum( delivery_fee((order["距离km"])) for order in orders)
print(total_fee)


# 任务3（附加）：计算每笔订单"配送费占订单金额的比例"（百分比，保留1位小数）
# 用字典推导生成 {订单号: "X%"} 
# 预期：{'M001': '14.3%', 'M002': '10.3%', 'M003': '54.5%', ...}

fee_ratio = {order["订单号"]:f"{delivery_fee(order['距离km'])/order['金额']*100:.1f}%" for order in orders}
print(fee_ratio)


46
{'M001': '14.3%', 'M002': '10.3%', 'M003': '54.5%', 'M004': '6.7%', 'M005': '9.1%', 'M006': '2.9%'}


---

## ⚠️ 新手最容易踩的坑

### 坑1：默认参数用可变对象——经典bug

❌ 错误写法：
```python
def add_order(order, orders_list=[]):
    orders_list.append(order)
    return orders_list

print(add_order("ORD001"))  # ['ORD001'] —— OK
print(add_order("ORD002"))  # ['ORD001', 'ORD002'] —— 😱怎么多了一个！
```

✅ 正确写法：
```python
def add_order(order, orders_list=None):
    if orders_list is None:
        orders_list = []       # 每次调用时创建全新的空list
    orders_list.append(order)
    return orders_list

print(add_order("ORD001"))  # ['ORD001']
print(add_order("ORD002"))  # ['ORD002'] —— 这才是我们想要的
```

**为什么错**：默认参数 `[]` 在 `def` 定义时只创建**一次**。之后每次调用如果不传，用的都是同一个列表对象——所有调用者共享了这个列表。就像全班共用一个垃圾桶，你扔的垃圾别人也能看见。

---

### 坑2：变量名覆盖内置函数

❌ 错误写法：
```python
sum = 0              # 把 sum 定义成了普通变量
for o in orders:
    sum += o['金额']
print(sum)           # OK，打印数字
total = sum(orders)  # 😱 TypeError: 'int' object is not callable!
```

✅ 正确写法：
```python
total = 0            # 变量名避开内置函数名
for o in orders:
    total += o['金额']
result = sum(o['金额'] for o in orders)  # sum函数还能用
```

**为什么错**：Python 不阻止你覆盖内置函数名。一旦 `sum = 0`，在当前作用域里 `sum` 就是一个 int，不再是内置求和函数。LEGB 查找顺序：先找到 L（你的局部变量 `sum=0`），就不会去 B（内置 `sum()`）找了。

> **避坑口诀**：`sum`, `list`, `dict`, `set`, `max`, `min`, `id`, `type`, `input`, `print` —— 这些都是内置函数名，永远不要用它们做变量名。

---

### 坑3：忘了return——函数返回None

❌ 错误写法：
```python
def calc_total(orders):
    total = 0
    for o in orders:
        total += o['金额']
    print(total)        # 只打印，没有return

result = calc_total(orders)
print(result)           # None —— 😱
print(result + 100)     # TypeError!
```

✅ 正确写法：
```python
def calc_total(orders):
    total = 0
    for o in orders:
        total += o['金额']
    return total         # 返回计算结果

result = calc_total(orders)
print(result)            # 547 —— 有值可用了
```

**为什么错**：没有 return 的函数自动返回 `None`。`print()` 只是把值显示在屏幕上，不能把值交给下一个计算步骤。就像你在计算器上算出一个数，看了一眼就清零了——没记下来。

---

### 坑4：推导式变量泄漏（Python 3.8及以前）

❌ Python 2/3.7 及以前——变量泄漏：
```python
x = 10
squares = [x**2 for x in range(5)]
print(x)  # 4 —— 😱全局x被推导式的循环变量覆盖了！
```

✅ Python 3.8+ 已修复——推导式有独立作用域：
```python
x = 10
squares = [x**2 for x in range(5)]
print(x)  # 10 —— 外面x不受影响
```

**为什么错**：Python 3.8 以前，推导式的循环变量会"泄漏"到外部作用域。如果你用的 Python 版本较老（< 3.8），注意这个坑。建议升级到 Python 3.9+。

> 你用 `python --version` 查一下版本。如果 ≥ 3.8，这个坑对你无效——但面试可能会考。


---

## 📝 参考答案

> **重要**：先自己写完所有练习再看答案。看着答案觉得自己"会了"是最大的幻觉。

### 概念1小练：count_big_orders

```python
def count_big_orders(orders, threshold):
    """统计金额大于threshold的订单数量"""
    count = 0
    for order in orders:
        if order['金额'] > threshold:
            count += 1
    return count

# 测试
print(count_big_orders(orders, 500))  # 2 (ORD002:680, ORD003:520)
print(count_big_orders(orders, 300))  # 3 (ORD001:299, ORD002:680, ORD003:520)
```

### 概念2小练：filter_orders

```python
def filter_orders(orders, city=None, min_amount=0):
    """按城市和/或最低金额筛选订单"""
    result = []
    for o in orders:
        if city and o['城市'] != city:
            continue
        if o['金额'] < min_amount:
            continue
        result.append(o)
    return result

# 测试
print(len(filter_orders(orders, city="北京")))           # 3
print(len(filter_orders(orders, min_amount=500)))         # 2
print(len(filter_orders(orders, city="北京", min_amount=500))) # 1
```

### 概念3小练：作用域bug修复

```python
# 方案1：用global声明
total_sales = 0
def add_sales(amount):
    global total_sales
    total_sales = total_sales + amount
    return total_sales

# 方案2：不用全局变量（推荐）
def add_sales(current_total, amount):
    return current_total + amount

total_sales = 0
total_sales = add_sales(total_sales, 100)  # 把返回值赋回给total_sales
total_sales = add_sales(total_sales, 200)
print(total_sales)  # 300
```

### 概念4小练：推导式

```python
# 任务1
big_ids = [o['订单号'] for o in orders if o['金额'] > 100]
print(big_ids)  # ['ORD001', 'ORD002', 'ORD003']

# 任务2
bj_amounts = [o['金额'] for o in orders if o['城市'] == '北京']
print(bj_amounts)  # [299, 520]

# 任务3
big_dict = {o['订单号']: o['金额'] for o in orders if o['金额'] >= 300}
print(big_dict)  # {'ORD002': 680, 'ORD003': 520}
```

### 基础练习1：calc_total

```python
def calc_total(orders):
    """计算订单list的总金额"""
    total = 0
    for order in orders:
        total += order['金额']
    return total

print(calc_total(orders))  # 1112
```

### 基础练习2：judge_order

```python
def judge_order(amount, threshold=500):
    """判断订单大小"""
    if amount > threshold:
        return "大单"
    elif amount == threshold:
        return "持平"
    else:
        return "小单"

# 调用测试
print(judge_order(680))                      # 大单
print(judge_order(500))                      # 持平
print(judge_order(299))                      # 小单
print(judge_order(299, threshold=300))       # 小单
```

### 基础练习3：列表推导

```python
# 任务1
digital_ids = [o['订单号'] for o in orders if o['品类'] == '数码']
print(digital_ids)  # ['ORD001', 'ORD003', 'ORD005', 'ORD007']

# 任务2
big_total = sum([o['金额'] for o in orders if o['金额'] > 500])
print(big_total)  # 1200

# 任务3
bj_sh = {o['订单号']: o['城市'] for o in orders if o['城市'] in ('北京', '上海')}
print(bj_sh)  # {'ORD001': '北京', 'ORD002': '上海', 'ORD004': '北京', 'ORD007': '上海'}
```

### 进阶练习1：玩家排行榜

```python
# 任务1
by_level = sorted(players, key=lambda p: p['等级'], reverse=True)
print([p['昵称'] for p in by_level])

# 任务2：多级排序——lambda返回tuple
by_both = sorted(players, key=lambda p: (-p['等级'], -p['充值额']))
print([p['昵称'] for p in by_both])
# ['疾风刺客', '圣光骑士', '冰霜法师', '风暴弓手', '暗影剑客']

# 任务3
vip = [p['昵称'] for p in players if p['等级'] >= 60 and p['充值额'] >= 300]
print(vip)  # ['冰霜法师', '圣光骑士']
```

### 进阶练习2：出行数据分析

```python
# 任务1
total = sum(filter(lambda a: a > 50, map(lambda t: t['金额'], trips)))
print(total)  # 253

# 任务2
per_km = list(map(lambda t: round(t['金额'] / t['里程km'], 1), trips))
print(per_km)  # [6.7, 6.1, 7.1, 6.5, 6.9, 6.6]

# 任务3解释：
# map/filter返回的是迭代器（惰性求值），不立即计算
# print(map(...)) 会打印 <map object at 0x...> 而不是结果
# sum()、list()、for 等会触发迭代器求值，所以sum()外面不需要list()
```

### 挑战练习：重构参考方案

```python
def calc_stats(orders):
    """统计订单：返回(总金额, 订单数, 最大单金额)"""
    total = 0
    count = 0
    max_amount = 0
    for o in orders:
        total += o['金额']
        count += 1
        if o['金额'] > max_amount:
            max_amount = o['金额']
    return total, count, max_amount

def filter_orders(orders, city=None, category=None):
    """筛选订单：按城市和/或品类过滤"""
    result = []
    for o in orders:
        if city and o['城市'] != city:
            continue
        if category and o['品类'] != category:
            continue
        result.append(o)
    return result

def summarize(orders, label):
    """统计并打印一组订单的汇总信息"""
    total, count, max_amt = calc_stats(orders)
    print(f"{label}: 总数{count}, 总额{total}, 最大单{max_amt}")

# 重构后的调用——一行一个场景
summarize(orders, "全部")
summarize(filter_orders(orders, city="北京"), "北京")
summarize(filter_orders(orders, category="数码"), "数码")
```

### 迁移练习：外卖配送费

```python
# 任务1
def delivery_fee(distance):
    """计算配送费"""
    if distance < 3:
        return 5
    elif distance <= 5:   # 3.0到5.0
        return 8
    else:
        return 12

# 任务2
total_fee = sum(delivery_fee(o['距离km']) for o in orders)
print(total_fee)  # 50

# 任务3
fee_ratio = {
    o['订单号']: f"{delivery_fee(o['距离km']) / o['金额'] * 100:.1f}%"
    for o in orders
}
print(fee_ratio)
# {'M001': '14.3%', 'M002': '10.3%', 'M003': '54.5%', ...}
```

---

## 🩺 偏差诊断

| 如果你的输出/报错是... | 你可能的误解是... | 补救 |
|----------------------|-----------------|------|
| `NameError: name 'xxx' is not defined` | 把函数内部的局部变量当成了全局变量 | 回到 Cell 5b 重看"作用域 LEGB"段，注意函数内变量外面访问不到 |
| 函数返回 `None` 或 `NoneType` 报错 | 写了 `print()` 忘了写 `return` | 回到 Cell 3b 重看第④段"为什么 return 不能省"；print 是显示，return 是交还结果 |
| `TypeError: 'int' object is not callable` | 你用 `sum`/`list`/`max` 做了变量名，覆盖了内置函数 | 回到 Cell 14 看坑2；把你用的变量名改成不冲突的 |
| 默认参数行为奇怪——第二次调用带着第一次的数据 | 默认参数用了可变对象（list/dict），所有调用共享同一个对象 | 回到 Cell 14 看坑1；用 `None` + 函数内初始化的模式 |
| 推导式结果不是list | 忘了用 `[]` 包起来——写成 `x for x in lst` 而不是 `[x for x in lst]` | 推导式必须包在 `[]` / `{}` / `{k:v}` 里；回到 Cell 6c 看语法 |
| `map`/`filter` 打印出来是 `<map object>` | 不知道 map/filter 返回迭代器，不是 list | 回到 Cell 6b 看第④段"map/filter 返回迭代器而非 list"；用 `list()` 转换 |
| `UnboundLocalError: local variable referenced before assignment` | 在函数里想读全局变量，但又在函数里给它赋值了（Python 认为它是局部变量）| 回到 Cell 5c 看代码示例；要么用 `global` 声明，要么用参数传入 |


In [11]:
# ========== 🔙 回到开头的问题 ==========

# 还记得早上的问题吗？小王给你三份订单数据，你复制粘贴了三遍
# 现在用今天学的函数，把统计逻辑封装起来：

last_week = [
    {"订单号": "LW001", "金额": 299},
    {"订单号": "LW002", "金额": 680},
    {"订单号": "LW003", "金额": 159},
]

this_week = [
    {"订单号": "TW001", "金额": 520},
    {"订单号": "TW002", "金额": 89},
    {"订单号": "TW003", "金额": 45},
]

promo = [
    {"订单号": "PM001", "金额": 1200},
    {"订单号": "PM002", "金额": 350},
]

# 任务：定义函数 analyze_orders(orders, label)
# 输入一个订单list和标签（如"上周"），打印：
#   "{label}: 总金额={total}, 订单数={count}, 大单数={big_count}"
# 然后对三份数据各调用一次
# 大单标准：>500元

# 你的代码（如果做不出→回到 Cell 3b-3c 重看函数定义，回到 Cell 4b-4c 重看默认参数）：

def analyze_orders(orders, label):
    """
    分析订单数据，打印总金额、订单数、大单数
    """
    total = sum(order["金额"] for order in orders)
    count = len(orders)
    big_count = sum(1 for order in orders if order["金额"] > 500)
    print(f"{label}: 总金额={total}, 订单数={count}, 大单数={big_count}")
    return total, count, big_count
print(analyze_orders(last_week, "上周"))
print(analyze_orders(this_week, "本周"))
print(analyze_orders(promo, "促销"))




# 预期输出：
# 上周: 总金额=1138, 订单数=3, 大单数=1
# 本周: 总金额=654, 订单数=3, 大单数=1
# 促销: 总金额=1550, 订单数=2, 大单数=1


上周: 总金额=1138, 订单数=3, 大单数=1
(1138, 3, 1)
本周: 总金额=654, 订单数=3, 大单数=1
(654, 3, 1)
促销: 总金额=1550, 订单数=2, 大单数=1
(1550, 2, 1)


---

## ✅ 今天你学会了吗？

- [ ] **初级（L2：能用）**：不看笔记，独立完成基础练习1-3
  没过 → 回看对应概念的 cell b-c，重做基础练习

- [ ] **中级（L3：能讲）**：下面写一段200字以内的白话，讲给不懂技术的同事听：
  
  > "函数是干什么的？参数有哪几种？什么时候用 lambda 而不是 def？"
  
  *你的回答（在这里写）：*
  
  
  
  
  
  
  没过 → 你写不出来说明停留在L2，回到 Cell 3b/4b/6b 看第④段

- [ ] **高级（L4：能判断）**：迁移练习独立完成了吗？换一个你从来没见过的场景，
  你能自己判断该用今天学的哪个武器吗？
  没过 → 补做迁移练习，不许查笔记

---

### 📋 概念掌握检查表（对照今日知识地图逐项检验）

**函数定义与调用**
- [ ] 能用 `def` 定义函数（括号+冒号+缩进体）
- [ ] 能解释"定义时不执行，调用时才执行"（记菜谱 vs 做菜）
- [ ] 能用 `return` 返回值，知道没有return返回None
- [ ] 能用 `return a, b` 返回多个值（知道实际返回tuple）
- [ ] 能给函数写文档字符串 `"""做什么"""`

**参数全系**
- [ ] 能写位置参数并正确传参
- [ ] 能写默认参数，且默认值放在必选参数后面
- [ ] 能用关键字参数换顺序传参
- [ ] 能解释为什么默认参数不能用可变对象（list/dict）
- [ ] 能用 `*args` 接收任意数量位置参数

**作用域 LEGB**
- [ ] 能说出 LEGB 四个字母分别代表什么
- [ ] 知道函数内变量外面访问不到
- [ ] 知道函数里读全局变量不需要 `global`，改需要
- [ ] 能解释"函数里 `x=10` 创建的是局部变量"

**lambda & 函数式工具**
- [ ] 能写 lambda 表达式：`lambda x: x['字段']`
- [ ] 能区分什么时候用 lambda（<1行，一次性的）什么时候用 def（多行，要复用）
- [ ] 能用 `map(func, iterable)` 对每个元素做操作
- [ ] 能用 `filter(func, iterable)` 筛出符合条件的
- [ ] 能用 `sorted(iterable, key=lambda)` 自定义排序
- [ ] 知道 map/filter 返回迭代器（惰性求值），需 `list()` 转换

**推导式**
- [ ] 能写列表推导：`[表达式 for x in 序列 if 条件]`
- [ ] 能写字典推导：`{k: v for k,v in d.items() if 条件}`
- [ ] 能写集合推导：`{x for x in 序列 if 条件}`
- [ ] 能讲出"推导式比 map+filter 好在哪"（可读性、更Pythonic）
- [ ] 知道 Python 3.8+ 推导式变量不会泄漏


---

## 🔮 明天预告

**明天你将能解决**：小王丢给你一份脏CSV文件——有乱码、有空行、金额那一列有些是汉字"待定"——你能不能把它读进来、洗干净、写回一份干净的CSV？

**今天学的函数** + **明天的文件IO** = 从"读文件 → 用函数清洗 → 写文件"的完整数据处理流水线。

---

### 🗺️ 今天的概念在本周知识网络中的位置

- **今天 → 明天（D4 文件IO）**：读文件 → 用函数处理数据 → 写文件。函数是整个数据流水线的"加工车间"——没有函数，你的文件处理代码就是一长串难以维护的脚本
- **今天 → 后天（D5 模块化）**：单文件太长 → 函数太多 → 按职责拆成不同 `.py` 模块。`def` 是模块的基本组成单位
- **今天 → W2 pandas**：pandas 的 `df.apply(func)` 和 `df.pipe(func)` 直接对应今天学的函数式编程思维——对数据框的每一行/列套用一个函数
- **今天 → D6 周项目**：你的项目代码里，函数就是你自制的工具库—— `calc_stats()`、`filter_orders()`、`summarize()`，它们组合在一起完成整个分析任务
